In [2]:
"""
Apple Health 数据提取脚本
========================
使用方法：
1. 把这个脚本放到 apple_health_export 文件夹里（和 export.xml 同目录）
2. 双击运行，或在命令行执行: python extract_health_data.py
3. 脚本会在同目录生成一个 health_summary 文件夹，里面有多个 CSV 文件
4. 把 health_summary 文件夹压缩成 zip 上传给 Claude 即可
"""

import xml.etree.ElementTree as ET
import csv
import os
from collections import defaultdict
from datetime import datetime

# === 配置 ===
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
XML_PATH = os.path.join(SCRIPT_DIR, "export.xml")
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "health_summary")

# 我们关心的数据类型
RECORD_TYPES = {
    # 心率相关
    "HKQuantityTypeIdentifierHeartRate": "heart_rate",
    "HKQuantityTypeIdentifierRestingHeartRate": "resting_heart_rate",
    "HKQuantityTypeIdentifierHeartRateVariabilitySDNN": "heart_rate_variability",
    "HKQuantityTypeIdentifierWalkingHeartRateAverage": "walking_heart_rate",
    # 运动 & 步数
    "HKQuantityTypeIdentifierStepCount": "step_count",
    "HKQuantityTypeIdentifierDistanceWalkingRunning": "distance_walking_running",
    "HKQuantityTypeIdentifierActiveEnergyBurned": "active_energy",
    "HKQuantityTypeIdentifierBasalEnergyBurned": "basal_energy",
    "HKQuantityTypeIdentifierAppleExerciseTime": "exercise_time",
    "HKQuantityTypeIdentifierAppleStandTime": "stand_time",
    "HKQuantityTypeIdentifierFlightsClimbed": "flights_climbed",
    # 睡眠
    "HKCategoryTypeIdentifierSleepAnalysis": "sleep",
    # 血氧
    "HKQuantityTypeIdentifierOxygenSaturation": "blood_oxygen",
    # 体温
    "HKQuantityTypeIdentifierBodyTemperature": "body_temperature",
    "HKQuantityTypeIdentifierAppleSleepingWristTemperature": "wrist_temperature",
    # 呼吸
    "HKQuantityTypeIdentifierRespiratoryRate": "respiratory_rate",
    # 噪音
    "HKQuantityTypeIdentifierEnvironmentalAudioExposure": "noise_exposure",
    "HKQuantityTypeIdentifierHeadphoneAudioExposure": "headphone_audio",
    # 身体数据
    "HKQuantityTypeIdentifierBodyMass": "body_mass",
    "HKQuantityTypeIdentifierHeight": "height",
    "HKQuantityTypeIdentifierBodyMassIndex": "bmi",
    "HKQuantityTypeIdentifierBodyFatPercentage": "body_fat",
    # VO2 Max
    "HKQuantityTypeIdentifierVO2Max": "vo2_max",
}

def parse_date(date_str):
    """解析日期字符串"""
    try:
        # Apple Health 格式: 2024-01-15 08:30:00 +0800
        return date_str[:19]  # 只取日期时间部分
    except:
        return date_str

def main():
    print("=" * 50)
    print("  Apple Health 数据提取工具")
    print("=" * 50)
    
    if not os.path.exists(XML_PATH):
        print(f"\n❌ 找不到 export.xml！")
        print(f"   请把这个脚本放到 apple_health_export 文件夹里")
        print(f"   当前查找路径: {XML_PATH}")
        input("\n按回车键退出...")
        return
    
    file_size = os.path.getsize(XML_PATH) / (1024 * 1024)
    print(f"\n📂 找到 export.xml ({file_size:.1f} MB)")
    print("⏳ 正在解析，大文件可能需要几分钟，请耐心等待...\n")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 用于存储数据的字典
    records = defaultdict(list)
    workouts = []
    ecg_records = []
    record_count = 0
    all_types_found = set()
    
    # 逐步解析 XML（节省内存）
    context = ET.iterparse(XML_PATH, events=("end",))
    
    for event, elem in context:
        
        # 处理 Record 元素
        if elem.tag == "Record":
            record_type = elem.get("type", "")
            all_types_found.add(record_type)
            
            if record_type in RECORD_TYPES:
                key = RECORD_TYPES[record_type]
                record = {
                    "date": parse_date(elem.get("startDate", "")),
                    "end_date": parse_date(elem.get("endDate", "")),
                    "value": elem.get("value", ""),
                    "unit": elem.get("unit", ""),
                    "source": elem.get("sourceName", ""),
                }
                records[key].append(record)
            
            record_count += 1
            if record_count % 500000 == 0:
                print(f"   已处理 {record_count:,} 条记录...")
            
            elem.clear()
        
        # 处理 Workout 元素
        elif elem.tag == "Workout":
            workout = {
                "type": elem.get("workoutActivityType", "").replace("HKWorkoutActivityType", ""),
                "date": parse_date(elem.get("startDate", "")),
                "end_date": parse_date(elem.get("endDate", "")),
                "duration_min": elem.get("duration", ""),
                "duration_unit": elem.get("durationUnit", ""),
                "total_distance": "",
                "distance_unit": "",
                "total_energy": "",
                "energy_unit": "",
            }
            
            # 提取统计数据
            for stat in elem.findall("WorkoutStatistics"):
                stat_type = stat.get("type", "")
                if "Distance" in stat_type:
                    workout["total_distance"] = stat.get("sum", "")
                    workout["distance_unit"] = stat.get("unit", "")
                elif "EnergyBurned" in stat_type:
                    workout["total_energy"] = stat.get("sum", "")
                    workout["energy_unit"] = stat.get("unit", "")
            
            workouts.append(workout)
            elem.clear()
        
        # 处理心电图
        elif elem.tag == "Correlation" and "Electrocardiogram" in elem.get("type", ""):
            ecg = {
                "date": parse_date(elem.get("startDate", "")),
                "classification": "",
            }
            for meta in elem.findall("MetadataEntry"):
                if "Classification" in meta.get("key", ""):
                    ecg["classification"] = meta.get("value", "")
            ecg_records.append(ecg)
            elem.clear()
    
    print(f"\n✅ 解析完成！共处理 {record_count:,} 条记录")
    
    # === 写入 CSV 文件 ===
    print("\n📊 正在生成 CSV 文件...\n")
    
    files_created = []
    
    # 1. 各类健康记录
    for key, data in records.items():
        if not data:
            continue
        filepath = os.path.join(OUTPUT_DIR, f"{key}.csv")
        
        # 对于高频数据（如心率、步数），按天聚合一份摘要
        if key in ("heart_rate", "step_count", "active_energy", "basal_energy", 
                    "distance_walking_running", "noise_exposure", "headphone_audio",
                    "exercise_time", "stand_time", "flights_climbed"):
            # 保存每日汇总版本
            daily = defaultdict(list)
            for r in data:
                day = r["date"][:10]
                try:
                    daily[day].append(float(r["value"]))
                except (ValueError, TypeError):
                    pass
            
            summary_path = os.path.join(OUTPUT_DIR, f"{key}_daily.csv")
            with open(summary_path, "w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                if key in ("step_count", "active_energy", "basal_energy", 
                           "distance_walking_running", "exercise_time", 
                           "stand_time", "flights_climbed"):
                    writer.writerow(["date", "total", "unit"])
                    for day in sorted(daily.keys()):
                        vals = daily[day]
                        writer.writerow([day, f"{sum(vals):.2f}", data[0]["unit"]])
                else:
                    writer.writerow(["date", "min", "max", "avg", "count", "unit"])
                    for day in sorted(daily.keys()):
                        vals = daily[day]
                        writer.writerow([
                            day,
                            f"{min(vals):.2f}",
                            f"{max(vals):.2f}",
                            f"{sum(vals)/len(vals):.2f}",
                            len(vals),
                            data[0]["unit"]
                        ])
            
            size = os.path.getsize(summary_path) / 1024
            files_created.append((f"{key}_daily.csv", len(daily), size))
            print(f"   ✅ {key}_daily.csv - {len(daily)} 天的数据 ({size:.1f} KB)")
        
        else:
            # 低频数据直接保存全部记录
            with open(filepath, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=["date", "end_date", "value", "unit", "source"])
                writer.writeheader()
                writer.writerows(data)
            
            size = os.path.getsize(filepath) / 1024
            files_created.append((f"{key}.csv", len(data), size))
            print(f"   ✅ {key}.csv - {len(data)} 条记录 ({size:.1f} KB)")
    
    # 2. 运动记录
    if workouts:
        filepath = os.path.join(OUTPUT_DIR, "workouts.csv")
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=[
                "type", "date", "end_date", "duration_min", "duration_unit",
                "total_distance", "distance_unit", "total_energy", "energy_unit"
            ])
            writer.writeheader()
            writer.writerows(workouts)
        
        size = os.path.getsize(filepath) / 1024
        files_created.append(("workouts.csv", len(workouts), size))
        print(f"   ✅ workouts.csv - {len(workouts)} 次运动 ({size:.1f} KB)")
    
    # 3. 心电图记录
    if ecg_records:
        filepath = os.path.join(OUTPUT_DIR, "ecg.csv")
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["date", "classification"])
            writer.writeheader()
            writer.writerows(ecg_records)
        
        size = os.path.getsize(filepath) / 1024
        files_created.append(("ecg.csv", len(ecg_records), size))
        print(f"   ✅ ecg.csv - {len(ecg_records)} 条心电图 ({size:.1f} KB)")
    
    # 4. 数据概览
    summary_path = os.path.join(OUTPUT_DIR, "_overview.txt")
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("Apple Health 数据概览\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"总记录数: {record_count:,}\n")
        f.write(f"运动记录: {len(workouts)}\n")
        f.write(f"心电图记录: {len(ecg_records)}\n\n")
        
        f.write("提取的数据文件:\n")
        for fname, count, size in files_created:
            f.write(f"  - {fname}: {count} 条 ({size:.1f} KB)\n")
        
        f.write(f"\n发现的所有数据类型 ({len(all_types_found)} 种):\n")
        for t in sorted(all_types_found):
            short = t.replace("HKQuantityTypeIdentifier", "").replace("HKCategoryTypeIdentifier", "")
            collected = "✅" if t in RECORD_TYPES else "⬜"
            f.write(f"  {collected} {short}\n")
    
    # 计算总大小
    total_size = sum(
        os.path.getsize(os.path.join(OUTPUT_DIR, f))
        for f in os.listdir(OUTPUT_DIR)
    ) / (1024 * 1024)
    
    print(f"\n{'=' * 50}")
    print(f"🎉 完成！共生成 {len(files_created) + 1} 个文件")
    print(f"📁 输出目录: {OUTPUT_DIR}")
    print(f"📦 总大小: {total_size:.2f} MB")
    print(f"\n👉 下一步:")
    print(f"   1. 把 health_summary 文件夹压缩成 zip")
    print(f"   2. 上传给 Claude 分析")
    print(f"{'=' * 50}")
    
    input("\n按回车键退出...")

if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined